# Imports

In [0]:
# init
from pyspark.sql import functions as F
from pyspark.sql.functions import when, upper, lower, trim, col
from pyspark.sql.types import StringType, IntegerType

# Read data from bronze schema 

In [0]:
df = spark.table("workspace.bronze_schema.erp_loc_a101")
df.display()

# Explore the data

In [0]:
df.printSchema()

In [0]:
df.describe().display()

"""
    From the count, CNTRY has some missing values
    from min we can see that it aslo has a possible empty string
    To be explore in details later
"""

In [0]:
df.columns
"""
    Column names are not user frinedly and needs to be remapped
"""

## Check for white spaces

In [0]:
for field in df.schema:
    column_name = field.name
    column_data_type = field.dataType

    if isinstance(column_data_type, StringType):
        white_space_df = df.select("*").where(trim(col(column_name)) != col(column_name))
        white_space_count = white_space_df.count()


        print(f"{white_space_count} white spaces detected in column {column_name}")
        if white_space_count:
            white_space_df.display()

"""
    5 White space detected in CNTRY 
"""

## Check for Null Values

In [0]:
for column in df.columns:

    null_df = df.select("*").where(col(column).isNull())
    null_count = null_df.count()

    print(f"{null_count} null values detected in column {column}")
    if null_count:
        null_df.display()

"""
    Null values detected in CNTRY
"""

## Check for distinct values in the CNTRY column

In [0]:
df.select("CNTRY").distinct().display()

"""
    There are 3 variations of the United States of America
    White spaces 
    Short country codes. Name should be in full
    
"""

## Check of the CID column matches the crm_cust_info customer_key

In [0]:
# Read silver_crm_cust_info to match erp_loc_a101
silver_crm_cust_info_df = spark.table("workspace.silver_schema.crm_cust_info")
silver_crm_cust_info_df.display()

In [0]:
"""
    According to out Entity Relationship Diagram (ERD), the CID should match the customer_key from crm_cust_info
"""

# using a join to check for matches 

(

    df.alias("location").join(
        silver_crm_cust_info_df.alias("customer"),
        (col("location.CID") == col("customer.customer_key")),
        how = "left"
    )
).display()


"""
    No matches which mean there might be a problem with the joining keys.
    Looking at it, we can see that "-" is in the locations CID but not in the customer customer_key
    Lets take a closer look by smapling some data from each table
"""



In [0]:
silver_crm_cust_info_df.select("customer_key").limit(5).display()
df.select("CID").limit(5).display()

""" 
    We can clearly see the difference now
    sample data: AW00011000 != AW-00011000
    "-" should be removed from the erp_loc_a101 CID
"""

# Data Notes
- Missing values(Null) and whitespace in the CNTRY column
- CNTRY need remapping. Multiple variations of country found and Short code used in some countries

- Column names are not business user friendly
- CID key does not match customer_key in crm_cust_info table because CID has "-" in it. Hyphen needs to be removed.

# Data Transformation

## Removing "-" from the CID column

In [0]:
df = df.withColumn(
    "CID",
    F.regexp_replace(col("CID"), "-", "")
)

"""
    By removing the "-" from the CID column, we now have some matches.
"""

In [0]:
df.select("*").where(~(col("CID").isin(silver_crm_cust_info_df.select("customer_key")))).display()

In [0]:
silver_crm_cust_info_df.select("*").where(~(col("customer_key").isin(df.select("CID")))).display()

In [0]:
df.display()

## Remapping the CNTRY to more uniform names

In [0]:
df.select("CNTRY").distinct().display()

In [0]:
df = df.withColumn(
    "CNTRY",
    when(
        trim(upper(col("CNTRY"))).isin(['US', 'UNITED STATES', 'USA', 'UNITED STATES OF AMERICA']), 'United States of America'
    )
    .when(
        trim(upper(col("CNTRY"))).isin(['GERMANY', 'DE']), 
        'Germany'
    )
    .when(
        trim(upper(col("CNTRY"))).isin(['UK', 'UNITED KINGDOM']), 
        'United Kingdom'
    )
    .when(
        col("CNTRY").isNull() | (trim(col("CNTRY")) == ""), 
        "n/a"
    )
    .otherwise(trim(col("CNTRY")))
)

df.select("CNTRY").distinct().display()

## Column Rename

In [0]:
COLUMNS_RENAMED = {
    'CID':'customer_key',
    'CNTRY':'customer_country'
}

df = df.withColumnsRenamed(COLUMNS_RENAMED)

df.display()

# Write Transformed data into the silver_schema

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver_schema.erp_loc_a101')

# Read data from the Silver_schema

In [0]:
silver_erp_loc_a101_df = spark.table("workspace.silver_schema.erp_loc_a101")
silver_erp_loc_a101_df.display()